# Demo 5 — Production Readiness

Walk through the local build/test loop before handing the platform team a container image and Bicep module.

**Prereqs**
- `pip install -r requirements.txt`
- A populated `.env`
- `az login`
- Docker (only needed for the optional image-build cell)

## 1. Inspect the agent factory and FastAPI surface

In [ ]:
print(open('app/agent.py').read())

In [ ]:
print(open('app/main.py').read())

## 2. Smoke-test the agent in-process

We import the factory and exercise it directly — same code path the FastAPI app uses, no HTTP overhead.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from app.agent import build_agent_client, create_agent

client, credential = build_agent_client()
agent = create_agent(client)

print(await agent.run("In one sentence: what does TP53 do?"))

await client.close()
await credential.close()

## 3. Boot the FastAPI app in the background and hit it

Demonstrates the HTTP surface the platform team will see.

In [ ]:
import asyncio, subprocess, sys, time, httpx

proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app", "--port", "8000", "--log-level", "warning"],
)

# Wait for /healthz
for _ in range(30):
    try:
        r = httpx.get("http://127.0.0.1:8000/healthz", timeout=1.0)
        if r.status_code == 200:
            print("healthz:", r.json())
            break
    except Exception:
        pass
    time.sleep(1)

r = httpx.post(
    "http://127.0.0.1:8000/chat",
    json={"message": "Pull a couple of recent papers on IKZF1 in pediatric B-ALL."},
    timeout=60,
)
print("/chat status:", r.status_code)
print(r.json())

In [ ]:
proc.terminate()
proc.wait(timeout=10)
print("server stopped")

## 4. Inspect the Dockerfile, Container App Bicep, and APIM policy

These are the three artifacts you hand the platform team.

In [ ]:
print('--- Dockerfile ---')
print(open('Dockerfile').read())
print('--- infra/container-app.bicep ---')
print(open('infra/container-app.bicep').read())
print('--- infra/apim-policy.xml ---')
print(open('infra/apim-policy.xml').read())

## 5. Build & run the image locally (optional)

Run these in a shell — they need Docker. They're written out as commands rather than executed so the notebook stays Docker-optional.

```powershell
docker build -t lit-triage-agent:dev .
docker run --rm -p 8000:8000 --env-file .env lit-triage-agent:dev
```

## 6. Hand off

Once the image is in the institutional ACR and your CI tags it, PR the
Bicep + APIM policy into the platform team's repos. The agent then
lives in the AI Gateway catalog and downstream consumers can subscribe.